# Training a Neural Network in PyTorch
***

## General overview

We need to instantiate a few objects listed below. These objects define how the model learns from data — the __model__ predicts, the __criterion__ evaluates, the __optimizer__ improves, and the __dataloader__ keeps the data flowing:

* __Model__: Defines the architecture of the neural network (layers, activations, etc.). It transforms input data into predictions. Created by subclassing `torch.nn.Module`
* __DataLoader__: Provides an efficient way to feed data to the model in batches, with optional shuffling and parallel loading. It wraps around datasets for training and testing
* __Criterion (Loss Function)__: Measures how far the model’s predictions are from the target values. It’s used to guide the learning process by computing an error signal
* __Optimizer__: Updates the model’s parameters based on the gradients computed from the loss. It implements the learning rule (e.g., SGD, Adam)

```mermaid
flowchart LR

A[(Raw data)] ==> B[Features<br/>Target arrays]
B ==>C{Splitting}
C ==>D[Validation Dataset] ==>F[Validation DataLoader]
C ==>E[Train Dataset] ==>G[Train DataLoader] ==> Z
subgraph Z [Training routine]
    direction LR
    H[Optimizer] -->K
    I[Criterion] -->K
    J[Model] -->K(Training loop)
end
```

## Requirements

Let's check the installation by importing the essential pytorch modules/sub-modules:

In [1]:
import torch                                               # main PyTorch library for tensor computation
import torch.nn as nn                                      # building blocks for creating and training neural networks
import torch.optim as optim                                # implementation of various optimization algorithms
from torch.utils.data import Dataset, DataLoader           # dataset utilities

## 1. Loading raw data and feature scaling

Feature scaling is a method used to normalize the range of independent variables or features of data.
In data processing, it is also known as data normalization and is generally performed during the data preprocessing step.

* __Normalization__: scales features to a specific range
* __Standardization__: transforms data to have a mean of 0 and a standard deviation of 1

| __Normalization__ |   |   | __Standardization__ |
|-------------------|---|---|---------------------|
| $ X \to \dfrac{X - X_{min}}{X_{max} - X_{min}}$ |   |   | $ X \to \dfrac{X-\mu}{\sigma}$ |

where $\mu$ and $\sigma$ are the mean and standard deviation of the features.

In [30]:
nfeatures = 4
ntargets  = 1
nexamples = 100

X = torch.rand(nexamples,nfeatures) ## Features
y = torch.ones(nexamples,noutputs)  ## Targets

print("Features array: ", X.shape)
print("Target vector: ", y.shape)

Features array:  torch.Size([100, 4])
Target vector:  torch.Size([100, 1])


## 2. Create Dataset

In [31]:
class myDataset(Dataset):
    def __init__(self, X, y, transform = None):
        """
        X: NumPy array of shape (n_samples, n_features)
        y: NumPy array of shape (n_samples, 1)
        """
        self.X = X
        self.y = y
        self.transform = transform

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        if self.transform:
            x = self.transform(x)
        return x, self.y[idx]

In [32]:
dataset = myDataset(X,y)
len(dataset)

100

In [33]:
# First examples
X, y = dataset[0]
X, y

(tensor([0.7414, 0.2247, 0.7554, 0.0274]), tensor([1.]))

## 3. Create DataLoader

In [34]:
# Create a data loader with batch_size of 16
loader = DataLoader(dataset, batch_size=16, shuffle=True)

In [35]:
for xb, yb in loader:
    print("**** Mini-batch of features: \n", xb)
    print("**** Mini-batch of targets: \n", yb)
    break

**** Batch features: 
 tensor([[0.1470, 0.4608, 0.2108, 0.7509],
        [0.6372, 0.1540, 0.5886, 0.7459],
        [0.9227, 0.4576, 0.9849, 0.3696],
        [0.3564, 0.0450, 0.0563, 0.4040],
        [0.8727, 0.0018, 0.6281, 0.6290],
        [0.4968, 0.3892, 0.1528, 0.8580],
        [0.3312, 0.1947, 0.4483, 0.4256],
        [0.1847, 0.4028, 0.6641, 0.7699],
        [0.2115, 0.8351, 0.3181, 0.3710],
        [0.7679, 0.1123, 0.7660, 0.1464],
        [0.0621, 0.3702, 0.3580, 0.5127],
        [0.3593, 0.4407, 0.6376, 0.3196],
        [0.0852, 0.7921, 0.2203, 0.9051],
        [0.0096, 0.1285, 0.7161, 0.0835],
        [0.5835, 0.6855, 0.3425, 0.4699],
        [0.1832, 0.7417, 0.2310, 0.9849]])
**** Batch targets: 
 tensor([[1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.]])


## 4. Define a model

**Multi-layer Perceptron (MLP)** is a supervised learning algorithm that learns
a function $f: R^m \rightarrow R^o$ by training on a dataset.
Given a set of features $X = \{x_1, x_2, ..., x_m\}$ and a target $y$, 
it can learn a non-linear function approximator for either classification or regression.

Overall, the perceptron unit is the function:

$$ f(x_0, \dots, x_n) = \varphi \left( \sum_{i=0}^{n} w_i x_i \right) $$

| Single perceptron unit   |
| ------------------------ |
| ![](figs/perceptron.svg) |

| Multilayer perceptron |
| ----------------------|
| ![](figs/mlp.svg)     |

[mlp]: https://scikit-learn.org/stable/modules/neural_networks_supervised.html "MLP"

In [38]:
# Define the model
class MultiLayerPerceptron(nn.Module):
    def __init__(self, nfeatures=4, nhidden=16):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(nfeatures, nhidden), # input layer -> hidden layer
            nn.ReLU(),
            nn.Linear(nhidden, 1),         # hidden layer -> output layer
            nn.ReLU(),
        )

    def forward(self, x):
        return self.model(x)

In [43]:
from torchsummary import summary

# Instantiate model
model = MultiLayerPerceptron()
summary(model, (nfeatures,))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                   [-1, 16]              80
              ReLU-2                   [-1, 16]               0
            Linear-3                    [-1, 1]              17
              ReLU-4                    [-1, 1]               0
Total params: 97
Trainable params: 97
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00
----------------------------------------------------------------


In [54]:
for xb, yb in loader:
    # Make a prediction
    yp = model(xb)
    print("**** Mini-batch of predictions: ", yp.shape)
    print("**** Mini-batch of targets: ", yb.shape)
    break

**** Mini-batch of predictions:  torch.Size([16, 1])
**** Mini-batch of targets:  torch.Size([16, 1])


## 5. Loss function

Let's define a loss function based on the mean squared error:

$$ MSE = \dfrac{1}{n} \sum_{i=1}^n (y_i-\hat{y}_i)^2 $$

In [45]:
# Creates a criterion that measures the mean squared error
criterion = nn.MSELoss()

In [47]:
# minibatch loop
for xb, yb in loader:
    # Make a prediction
    yp = model(xb)
    # Compute MSE loss
    loss = criterion(yp,yb)
    print("**** Averaged loss: ", loss)
    break

**** Averaged loss:  tensor(0.7579, grad_fn=<MseLossBackward0>)


## 6. Optimizer

In a method like _Stochastic Gradient Descent (SGD)_, the learning rate is multiplied by the gradient of the loss function to update the weights:

$$ w(t+1) = w(t) - \eta \dfrac{\partial L}{\partial w} (t) $$

where $w$ represents the weights and $\eta$ is the learning rate.

The learning rate is a key hyperparameter in neural networks that controls how quickly the model learns during training

In [48]:
# Create an optimizer
optimizer = optim.Adam(model.parameters(), lr=1E-3)

## A template for the training routine

In [49]:
def train_epoch(model, loader, criterion, optimizer):
    # Set training mode
    model.train()
    
    total_loss = 0.0
    
    for xb, yb in loader:
        # Make a prediction
        yp = model(xb)
        # Compute MSE loss
        loss = criterion(yp,yb)

        # Update gradients
        optimizer.zero_grad()
        loss.backward()

        # Update parameters
        optimizer.step()
        
        # Update total loss
        total_loss += loss.item()
    return total_loss / len(loader.dataset)

## Training loop

In [52]:
NEPOCHS = 500

for epoch in range(NEPOCHS):
    loss = train_epoch(model, loader, criterion, optimizer)
    if epoch%10 == 0:
        print(f"Epoch {epoch+1:03d} -> Train loss {loss:.4f}")

Epoch 001 -> Train loss 0.0000
Epoch 011 -> Train loss 0.0000
Epoch 021 -> Train loss 0.0000
Epoch 031 -> Train loss 0.0000
Epoch 041 -> Train loss 0.0000
Epoch 051 -> Train loss 0.0000
Epoch 061 -> Train loss 0.0000
Epoch 071 -> Train loss 0.0000
Epoch 081 -> Train loss 0.0000
Epoch 091 -> Train loss 0.0000
Epoch 101 -> Train loss 0.0000
Epoch 111 -> Train loss 0.0000
Epoch 121 -> Train loss 0.0000
Epoch 131 -> Train loss 0.0000
Epoch 141 -> Train loss 0.0000
Epoch 151 -> Train loss 0.0000
Epoch 161 -> Train loss 0.0000
Epoch 171 -> Train loss 0.0000
Epoch 181 -> Train loss 0.0000
Epoch 191 -> Train loss 0.0000
Epoch 201 -> Train loss 0.0000
Epoch 211 -> Train loss 0.0000
Epoch 221 -> Train loss 0.0000
Epoch 231 -> Train loss 0.0000
Epoch 241 -> Train loss 0.0000
Epoch 251 -> Train loss 0.0000
Epoch 261 -> Train loss 0.0000
Epoch 271 -> Train loss 0.0000
Epoch 281 -> Train loss 0.0000
Epoch 291 -> Train loss 0.0000
Epoch 301 -> Train loss 0.0000
Epoch 311 -> Train loss 0.0000
Epoch 32

## Performing an inference 

In [51]:
## Set inference mode
model.eval()

## Validation dataset
X_val = torch.rand((10,nfeatures))

with torch.no_grad():
    yp = model(X_val)
yp

tensor([[1.0025],
        [1.0003],
        [0.9992],
        [1.0007],
        [1.0026],
        [0.9977],
        [1.0009],
        [1.0008],
        [0.9975],
        [1.0020]])